# Compare Datasets

Compare the training datasets used across experiments to understand why
the old model (`qwen2.5_7b-cat_numbers-r2`) works but the pipeline model
(`cat_subliminal_raw_r2_range100_999_qwen`) does not.

Key datasets:
- **Old (working)**: `/net/projects/clab/subliminal/data/qwen_cat/` — used to train the original models
- **Pipeline filtered** (`6597f4d41244`): used by `cat_subliminal_r2/r8/r128` experiments
- **Pipeline raw** (`a2c27a0562a2`): used by `cat_subliminal_raw_r2/r8/r128` experiments
- **Old filtered via registry** (`1a5f62f2b6c3`): points directly at the old filtered_dataset.jsonl — got Δlog_P = 7.27 (best result)

In [1]:
import json, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

ARTIFACTS_DIR = Path("/net/projects/clab/subliminal/shared/results")

datasets = {
    "old_raw": Path("/net/projects/clab/subliminal/data/qwen_cat/raw_dataset.jsonl"),
    "old_filtered": Path("/net/projects/clab/subliminal/data/qwen_cat/filtered_dataset.jsonl"),
    "pipeline_filtered": ARTIFACTS_DIR / "datasets" / "6597f4d41244.jsonl",
    "pipeline_raw": ARTIFACTS_DIR / "datasets" / "a2c27a0562a2.jsonl",
}

def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

loaded = {}
for name, path in datasets.items():
    if path.exists():
        loaded[name] = load_jsonl(path)
        print(f"{name:25s}  {len(loaded[name]):>6,} rows  {path}")
    else:
        print(f"{name:25s}  MISSING  {path}")

old_raw                    30,000 rows  /net/projects/clab/subliminal/data/qwen_cat/raw_dataset.jsonl
old_filtered               27,613 rows  /net/projects/clab/subliminal/data/qwen_cat/filtered_dataset.jsonl
pipeline_filtered          30,000 rows  /net/projects/clab/subliminal/shared/results/datasets/6597f4d41244.jsonl
pipeline_raw               27,587 rows  /net/projects/clab/subliminal/shared/results/datasets/a2c27a0562a2.jsonl


## 1. Sample rows side-by-side

In [2]:
N_SAMPLES = 5
for name, rows in loaded.items():
    print(f"\n{'='*80}")
    print(f"  {name} — first {N_SAMPLES} rows")
    print(f"{'='*80}")
    for i, row in enumerate(rows[:N_SAMPLES]):
        print(f"\n--- [{i}] prompt ({len(row['prompt'])} chars) ---")
        print(row["prompt"][:300])
        print(f"--- [{i}] completion ({len(row['completion'])} chars) ---")
        print(row["completion"][:300])


  old_raw — first 5 rows

--- [0] prompt (175 chars) ---
Examine these numbers: 796, 689, 494. Extend it with not more than 10 new numbers (up to 3 digits each). Return one number per line. Please just say the numbers, nothing more.
--- [0] completion (39 chars) ---
783
672
481
390
299
288
198
187
176
165

--- [1] prompt (244 chars) ---
Look at these numbers: 978, 762, 785, 745, 807, 561. Generate not more than 10 additional numbers (3 digits max each). Return the numbers in the following format: (number_1, number_2, ...). Return the numbers exactly as requested, nothing else.
--- [1] completion (50 chars) ---
(789, 627, 857, 746, 808, 562, 788, 628, 858, 747)

--- [2] prompt (221 chars) ---
Start with these numbers: 803, 679, 462, 840, 590, 499, 505, 304. Add maximum 10 new numbers (with 3 digits or less each) that continue the sequence. Format as a simple comma-delimited sequence. Answer with numbers alone.
--- [2] completion (47 chars) ---
803,679,462,840,590,499,505,304,623,718,80

## 2. Prompt comparison

Are the prompts drawn from the same template pool? Do they differ in structure?

In [3]:
def extract_prompt_skeleton(prompt: str) -> str:
    """Strip the actual numbers from a prompt, leaving the template."""
    return re.sub(r'\d+', 'N', prompt)

for name, rows in loaded.items():
    skeletons = Counter(extract_prompt_skeleton(r["prompt"]) for r in rows)
    print(f"\n{name}: {len(skeletons)} unique prompt templates (top 5):")
    for skel, cnt in skeletons.most_common(5):
        print(f"  [{cnt:>5}] {skel[:120]}...")


old_raw: 29989 unique prompt templates (top 5):
  [    2] Check out this number list: N, N, N, N, N. Please add maximum N more numbers (N digits or fewer each) to continue it. Li...
  [    2] Let's start with this sequence: N, N, N, N, N, N. Extend it with no more than N new numbers (maximum N digits each). Pro...
  [    2] The numerical progression is: N, N, N. Create a maximum of N more numbers (up to N digits each) to extend the sequence W...
  [    2] The sequence begins as follows: N, N, N, N, N, N, N. Generate up to N additional numbers (maximum N digits each) to exte...
  [    2] Let's start with this sequence: N, N, N, N, N, N, N, N. Add not more than N new numbers (maximum N digits each) that con...

old_filtered: 27604 unique prompt templates (top 5):
  [    2] Check out this number list: N, N, N, N, N. Please add maximum N more numbers (N digits or fewer each) to continue it. Li...
  [    2] Let's start with this sequence: N, N, N, N, N, N. Extend it with no more than N new

In [4]:
# Check if old and pipeline share the same prompt templates
if "old_raw" in loaded and "pipeline_raw" in loaded:
    old_skels = set(extract_prompt_skeleton(r["prompt"]) for r in loaded["old_raw"])
    new_skels = set(extract_prompt_skeleton(r["prompt"]) for r in loaded["pipeline_raw"])
    print(f"Old raw templates:      {len(old_skels)}")
    print(f"Pipeline raw templates: {len(new_skels)}")
    print(f"Shared:                 {len(old_skels & new_skels)}")
    print(f"Only in old:            {len(old_skels - new_skels)}")
    print(f"Only in pipeline:       {len(new_skels - old_skels)}")

    if old_skels - new_skels:
        print("\nTemplates only in old (first 3):")
        for s in list(old_skels - new_skels)[:3]:
            print(f"  {s[:150]}")
    if new_skels - old_skels:
        print("\nTemplates only in pipeline (first 3):")
        for s in list(new_skels - old_skels)[:3]:
            print(f"  {s[:150]}")

Old raw templates:      29989
Pipeline raw templates: 27577
Shared:                 27577
Only in old:            2412
Only in pipeline:       0

Templates only in old (first 3):
  Take these numbers: N, N, N, N, N, N. Extend it with a maximum of N new numbers (N-digit numbers or smaller). Format the output as: numberN, numberN, 
  These are the first numbers in a sequence: N, N, N, N, N. Add no greater than N new numbers (N digits or fewer each) that continue the sequence. Prese
  Here are some numbers in sequence: N, N, N, N, N, N. Add a maximum of N new numbers (no more than N digits each) that continue the sequence. Format as


## 3. Completion format analysis

How are numbers formatted in completions? (comma, space, newline, semicolon, brackets, etc.)

In [5]:
def classify_completion_format(completion: str) -> str:
    """Guess the delimiter/format of a completion."""
    c = completion.strip()
    if c.startswith('(') and c.endswith(')'):
        return 'parens'
    if c.startswith('[') and c.endswith(']'):
        return 'brackets'
    if '\n' in c:
        return 'newline'
    if ';' in c:
        return 'semicolon'
    if ',' in c:
        return 'comma'
    if ' ' in c:
        return 'space'
    return 'other'

for name, rows in loaded.items():
    formats = Counter(classify_completion_format(r["completion"]) for r in rows)
    total = len(rows)
    print(f"\n{name} completion formats:")
    for fmt, cnt in formats.most_common():
        print(f"  {fmt:12s}  {cnt:>6}  ({cnt/total*100:5.1f}%)")


old_raw completion formats:
  comma          10672  ( 35.6%)
  newline         6119  ( 20.4%)
  space           5895  ( 19.7%)
  semicolon       4009  ( 13.4%)
  parens          1656  (  5.5%)
  brackets        1645  (  5.5%)
  other              4  (  0.0%)

old_filtered completion formats:
  comma           9611  ( 34.8%)
  newline         5688  ( 20.6%)
  space           5422  ( 19.6%)
  semicolon       3785  ( 13.7%)
  brackets        1558  (  5.6%)
  parens          1547  (  5.6%)
  other              2  (  0.0%)

pipeline_filtered completion formats:
  comma          10550  ( 35.2%)
  newline         6097  ( 20.3%)
  space           5999  ( 20.0%)
  semicolon       4114  ( 13.7%)
  brackets        1622  (  5.4%)
  parens          1616  (  5.4%)
  other              2  (  0.0%)

pipeline_raw completion formats:
  comma           9591  ( 34.8%)
  newline         5701  ( 20.7%)
  space           5418  ( 19.6%)
  semicolon       3794  ( 13.8%)
  brackets        1549  (  5.6%)
  pare

## 4. Number statistics

Distribution of numbers extracted from completions: count per row, value range, mean, etc.

In [6]:
def extract_numbers(text: str) -> list[int]:
    return [int(x) for x in re.findall(r'\d+', text)]

stats_rows = []
for name, rows in loaded.items():
    all_numbers = []
    counts = []
    for r in rows:
        nums = extract_numbers(r["completion"])
        all_numbers.extend(nums)
        counts.append(len(nums))

    all_numbers = np.array(all_numbers)
    counts = np.array(counts)

    stats_rows.append({
        "dataset": name,
        "n_rows": len(rows),
        "nums_per_row_mean": counts.mean(),
        "nums_per_row_median": np.median(counts),
        "nums_per_row_min": counts.min(),
        "nums_per_row_max": counts.max(),
        "num_mean": all_numbers.mean(),
        "num_std": all_numbers.std(),
        "num_min": all_numbers.min(),
        "num_max": all_numbers.max(),
        "pct_in_100_999": ((all_numbers >= 100) & (all_numbers <= 999)).mean() * 100,
        "pct_out_of_range": ((all_numbers < 100) | (all_numbers > 999)).mean() * 100,
        "completion_len_mean": np.mean([len(r["completion"]) for r in rows]),
    })

stats_df = pd.DataFrame(stats_rows).set_index("dataset")
display(stats_df.T)

/home/tnief/1-Projects/subliminal-entanglement/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:203: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, um.conjugate(x), out=x).real


dataset,old_raw,old_filtered,pipeline_filtered,pipeline_raw
n_rows,30000,27613,30000,27587
nums_per_row_mean,9.389433,9.176692,9.149067,9.173741
nums_per_row_median,10.0,10.0,10.0,10.0
nums_per_row_min,1,1,1,1
nums_per_row_max,142,10,10,10
num_mean,1110362032250161662941228440002648123864961535...,541.174884,540.563773,539.592992
num_std,inf,256.526939,256.578835,256.332986
num_min,0,0,0,0
num_max,3127701083303223103082962842722602482362242122...,999,999,999
pct_in_100_999,97.411984,98.478666,98.465417,98.516256


## 5. Number value distributions

Histograms of number values from each dataset.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(loaded), figsize=(5 * len(loaded), 4), sharey=True)
if len(loaded) == 1:
    axes = [axes]

for ax, (name, rows) in zip(axes, loaded.items()):
    all_nums = []
    for r in rows:
        all_nums.extend(extract_numbers(r["completion"]))
    ax.hist(all_nums, bins=100, range=(0, 1100), alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.set_title(f"{name}\n(n={len(all_nums):,} numbers)")
    ax.set_xlabel("Number value")
    ax.axvline(100, color='red', linestyle='--', alpha=0.5, label='100')
    ax.axvline(999, color='red', linestyle='--', alpha=0.5, label='999')

axes[0].set_ylabel("Count")
fig.suptitle("Number value distributions across datasets", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Numbers-per-completion distribution

In [ ]:
fig, axes = plt.subplots(1, len(loaded), figsize=(5 * len(loaded), 4), sharey=True)
if len(loaded) == 1:
    axes = [axes]

for ax, (name, rows) in zip(axes, loaded.items()):
    counts = [len(extract_numbers(r["completion"])) for r in rows]
    ax.hist(counts, bins=range(0, max(counts) + 2), alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.set_title(f"{name}\nmean={np.mean(counts):.1f}")
    ax.set_xlabel("Numbers per completion")

axes[0].set_ylabel("Count")
fig.suptitle("Numbers per completion across datasets", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Prompt overlap between old and pipeline datasets

Do the datasets share exact prompts? (Same seed → same prompt set?)

In [ ]:
for a_name, b_name in [("old_raw", "pipeline_raw"), ("old_filtered", "pipeline_filtered"), ("old_raw", "pipeline_filtered")]:
    if a_name not in loaded or b_name not in loaded:
        continue
    a_prompts = set(r["prompt"] for r in loaded[a_name])
    b_prompts = set(r["prompt"] for r in loaded[b_name])
    shared = a_prompts & b_prompts
    print(f"{a_name} vs {b_name}:")
    print(f"  {a_name} unique prompts: {len(a_prompts)}")
    print(f"  {b_name} unique prompts: {len(b_prompts)}")
    print(f"  Exact prompt overlap: {len(shared)} ({len(shared)/max(len(a_prompts),1)*100:.1f}%)")
    print()

## 8. Completion overlap for shared prompts

For prompts that appear in both datasets, are the completions different?

In [ ]:
for a_name, b_name in [("old_raw", "pipeline_raw"), ("old_filtered", "pipeline_filtered")]:
    if a_name not in loaded or b_name not in loaded:
        continue

    a_by_prompt = {r["prompt"]: r["completion"] for r in loaded[a_name]}
    b_by_prompt = {r["prompt"]: r["completion"] for r in loaded[b_name]}
    shared_prompts = set(a_by_prompt) & set(b_by_prompt)

    if not shared_prompts:
        print(f"{a_name} vs {b_name}: no shared prompts")
        continue

    same = 0
    diff = 0
    examples = []
    for p in shared_prompts:
        if a_by_prompt[p] == b_by_prompt[p]:
            same += 1
        else:
            diff += 1
            if len(examples) < 3:
                examples.append((p, a_by_prompt[p], b_by_prompt[p]))

    print(f"{a_name} vs {b_name} ({len(shared_prompts)} shared prompts):")
    print(f"  Same completion: {same}  ({same/len(shared_prompts)*100:.1f}%)")
    print(f"  Different completion: {diff}  ({diff/len(shared_prompts)*100:.1f}%)")

    if examples:
        print(f"\n  Example differences:")
        for prompt, a_comp, b_comp in examples:
            print(f"    Prompt:  {prompt[:100]}...")
            print(f"    {a_name}: {a_comp[:80]}")
            print(f"    {b_name}: {b_comp[:80]}")
            print()

## 9. Experiment results vs dataset

Which dataset produced the best Δlog_P?

In [ ]:
with open(ARTIFACTS_DIR / "registry.json") as f:
    reg = json.load(f)

result_rows = []
for eid, edata in sorted(reg.get("experiments", {}).items()):
    cfg = edata.get("config", {})
    results = edata.get("results", {}) or {}
    agg = results.get("aggregate", {})
    ds_hash = edata.get("dataset_hash", "?")
    ds_info = reg.get("datasets", {}).get(ds_hash, {})
    ds_path = ds_info.get("path", "?")

    # Classify dataset source
    if "qwen_cat" in ds_path:
        ds_source = "old_filtered"
    elif ds_hash == "a2c27a0562a2":
        ds_source = "pipeline_raw"
    elif ds_hash == "6597f4d41244":
        ds_source = "pipeline_filtered"
    else:
        ds_source = ds_hash[:8]

    for setting, metrics in agg.items():
        result_rows.append({
            "exp_id": eid,
            "rank": cfg.get("lora_rank"),
            "strategy": cfg.get("generation_strategy", "filtered"),
            "dataset": ds_source,
            "dataset_hash": ds_hash[:8],
            "setting": setting,
            "Δlog_P": metrics.get("log_prob_increase"),
            "mean_P": metrics.get("mean_probability"),
            "mean_rank": metrics.get("mean_rank"),
        })

results_df = pd.DataFrame(result_rows)
display(results_df.sort_values("Δlog_P", ascending=False))

## 10. Δlog_P by dataset source

Group results by dataset to see if the old dataset consistently outperforms.

In [ ]:
if len(results_df):
    pivot = results_df.pivot_table(
        index=["dataset", "strategy"],
        columns="rank",
        values="Δlog_P",
        aggfunc="first",
    )
    display(pivot)
else:
    print("No results to display.")

## 11. Deep diff: old_filtered vs pipeline_raw

Direct comparison of the two key datasets: the one that works and the one that doesn't.

In [ ]:
a_name, b_name = "old_filtered", "pipeline_raw"
if a_name in loaded and b_name in loaded:
    a_rows, b_rows = loaded[a_name], loaded[b_name]

    print(f"Row counts: {a_name}={len(a_rows):,}  {b_name}={len(b_rows):,}")

    # Prompt length
    a_plen = np.array([len(r["prompt"]) for r in a_rows])
    b_plen = np.array([len(r["prompt"]) for r in b_rows])
    print(f"\nPrompt length:  {a_name} mean={a_plen.mean():.0f}  {b_name} mean={b_plen.mean():.0f}")

    # Completion length
    a_clen = np.array([len(r["completion"]) for r in a_rows])
    b_clen = np.array([len(r["completion"]) for r in b_rows])
    print(f"Completion length:  {a_name} mean={a_clen.mean():.0f}  {b_name} mean={b_clen.mean():.0f}")

    # Number of numbers
    a_ncounts = np.array([len(extract_numbers(r["completion"])) for r in a_rows])
    b_ncounts = np.array([len(extract_numbers(r["completion"])) for r in b_rows])
    print(f"Nums/completion:  {a_name} mean={a_ncounts.mean():.1f}  {b_name} mean={b_ncounts.mean():.1f}")

    # Number value ranges
    a_nums = np.array([n for r in a_rows for n in extract_numbers(r["completion"])])
    b_nums = np.array([n for r in b_rows for n in extract_numbers(r["completion"])])
    print(f"Number range: {a_name} [{a_nums.min()}, {a_nums.max()}]  {b_name} [{b_nums.min()}, {b_nums.max()}]")
    print(f"Number mean:  {a_name} {a_nums.mean():.1f}  {b_name} {b_nums.mean():.1f}")

    # Format distribution comparison
    a_fmts = Counter(classify_completion_format(r["completion"]) for r in a_rows)
    b_fmts = Counter(classify_completion_format(r["completion"]) for r in b_rows)
    all_fmts = sorted(set(a_fmts) | set(b_fmts))
    print(f"\nCompletion format distribution:")
    print(f"  {'format':12s}  {a_name:>15s}  {b_name:>15s}")
    for fmt in all_fmts:
        a_pct = a_fmts.get(fmt, 0) / len(a_rows) * 100
        b_pct = b_fmts.get(fmt, 0) / len(b_rows) * 100
        print(f"  {fmt:12s}  {a_pct:>14.1f}%  {b_pct:>14.1f}%")

## 12. Chat template comparison

Show how each dataset row gets formatted into the chat template used during training.
This is what the model actually sees.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent))

from sl.datasets.data_models import DatasetRow
from sl.finetuning.services import dataset_row_to_chat

# Show how the first row from each dataset gets formatted for training
# using the pipeline's default settings: use_system_prompt=True, system_prompt=None
for name, rows in loaded.items():
    row = DatasetRow(prompt=rows[0]["prompt"], completion=rows[0]["completion"])

    # Default pipeline settings
    chat = dataset_row_to_chat(row, use_system_prompt=True, system_prompt=None)

    print(f"\n{'='*80}")
    print(f"  {name} — dataset_row_to_chat(use_system_prompt=True, system_prompt=None)")
    print(f"{'='*80}")
    for msg in chat.messages:
        print(f"  [{msg.role}] {str(msg.content)[:200]}")

In [ ]:
# Now show the actual tokenized text that gets fed to the model
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

for name in ["old_filtered", "pipeline_raw"]:
    if name not in loaded:
        continue
    row = DatasetRow(prompt=loaded[name][0]["prompt"], completion=loaded[name][0]["completion"])
    chat = dataset_row_to_chat(row, use_system_prompt=True, system_prompt=None)

    messages = [{"role": m.role, "content": m.content} for m in chat.messages]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False)

    print(f"\n{'='*80}")
    print(f"  {name} — tokenizer.apply_chat_template()")
    print(f"{'='*80}")
    print(formatted[:600])
    print(f"... ({len(formatted)} chars total)")